# Ch01 LLM Fundamentals — GPU Supplement

This notebook requires a CUDA GPU with ≥20 GB VRAM to run large local models.

**Experiments**:
1. Model scale comparison: phi3:mini (3.8B) vs llama3.2:70b (70B)
2. Lost-in-middle at scale: 32k token context with large model

**Requirements**: Ollama + GPU, models pre-pulled: `ollama pull llama3.2:70b`

In [ ]:
import torch

if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU detected — run the CPU notebook instead: notebook-exercise.ipynb\n"
        "To provision a GPU machine: see notes/06-ai-infrastructure/ch01-gpu-architecture/"
    )

device = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {device}  |  VRAM: {vram:.1f} GB")

assert vram >= 20, f"Need ≥20 GB VRAM for llama3.2:70b, got {vram:.1f} GB"
print("✓ GPU check passed")

## Experiment 1 — Model Scale & Temperature Consistency

Compare how phi3:mini (3.8B) vs llama3.2:70b (70B) behave at different temperatures.

**Hypothesis**: Larger models should show more consistent output at T=0 and more coherent variety at T=1.0.

In [ ]:
import ollama

prompt = "Explain why transformers replaced RNNs in NLP in one sentence."
models = ['phi3:mini', 'llama3.2:70b']
temperatures = [0.0, 1.0]

for model in models:
    print(f"\n{'='*60}")
    print(f"Model: {model}")
    print('='*60)

    for temp in temperatures:
        responses = []
        print(f"\nTemperature = {temp}")
        for i in range(5):
            response = ollama.generate(
                model=model,
                prompt=prompt,
                options={'temperature': temp, 'num_predict': 100}
            )
            responses.append(response['response'][:150])
            print(f"{i+1}. {response['response'][:150]}...")

        unique = len(set(responses))
        print(f"\nUnique responses: {unique}/5")

## Experiment 2 — Lost-in-Middle at 32k Context

Test whether llama3.2:70b (128k context window) degrades less than phi3:mini (4k) on long contexts.

**Setup**: Inject target fact at different positions in a 32k token document.

In [ ]:
import ollama

# Generate large filler (32k tokens ≈ 24k words ≈ 4000 sentences)
filler_sentence = "This is general background information about various topics in computer science and machine learning. "
filler = filler_sentence * 4000  # ~32k tokens

target = "CRITICAL FACT: The Chinchilla paper showed that models should be trained on 20 tokens per parameter for compute-optimal convergence."
question = "What did the Chinchilla paper show about compute-optimal training?"

positions = [0.0, 0.25, 0.5, 0.75, 1.0]
results = {}

print("Testing llama3.2:70b on 32k context...")

for pos in positions:
    hits = 0
    for run in range(3):
        # Inject at position
        idx = int(len(filler) * pos)
        context = filler[:idx] + " " + target + " " + filler[idx:]

        prompt = f"{context}\n\nQuestion: {question}\nAnswer:"

        response = ollama.generate(
            model='llama3.2:70b',
            prompt=prompt,
            options={'temperature': 0.0, 'num_predict': 100}
        )

        if "20 tokens" in response['response'] or "Chinchilla" in response['response']:
            hits += 1

    hit_rate = hits / 3 * 100
    results[pos] = hit_rate
    print(f"Position {pos:.0%}: {hits}/3 = {hit_rate:.0f}%")